In [1]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.layers import LSTM, Dense, Dropout
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.stattools import arma_order_select_ic
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset (2).csv


In [3]:
dataset = pd.read_csv("Cleaned Dataset.csv")

In [4]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [5]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Data splitting into Pre and Post COVID
pre_covid_df = dataset[dataset['Year'] < 2020].copy()
post_covid_df = dataset[dataset['Year'] >= 2020].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [7]:
# Check which columns have NaNs and how many
print("Missing Values Per Column (Pre-COVID)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column (Post-COVID)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column (Pre-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           8
dtype: int64

Missing Values Per Column (Post-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64


In [8]:
# Checking which rows are empty
print("Missing Log_CPI in Pre-COVID")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])
print("\n")
print("Missing Log_CPI in Post-COVID")
print(post_covid_df[post_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Pre-COVID
    Country  Year  Log_CPI
0  Bulgaria  2014      NaN
1   Croatia  2016      NaN
2    Cyprus  2014      NaN
3    Cyprus  2015      NaN
4    Cyprus  2016      NaN
5    Greece  2014      NaN
6    Greece  2015      NaN
7   Romania  2016      NaN


Missing Log_CPI in Post-COVID
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [9]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of Pre-COVID data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [10]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [11]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [12]:
# Define the target
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (243, 3)
Test Shape (PCs): (135, 3)


In [13]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France',
    'Germany', 'Hungary', 'Italy', 'Netherlands', 'Poland',
    'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece', 'Ireland',
    'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Portugal', 'Slovakia', 'Slovenia'
]

In [14]:
# Filter Training Data (Pre-COVID)
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data (Post-COVID)
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Proposed Model

**1. LSTM Model**

In [15]:
def run_lstm_model(X_train, y_train, X_test, y_test, country_name):

    # Scale features and target that shrinks the data between 0 and 1.
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    # Applies the same transformation for train and test data for features and target
    # For features
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    # For target
    # The scaler expects a 2D array, so we turn a flat list of targets into a column.
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
    y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

    # Reshape for LSTM (samples, timestamp, features) for training
    # By timestamps = 1, to tell the model that each prediction is based on the current year's datapoint
    X_train_3D = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_3D = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    # LSTM Architecture
    model = Sequential([
        LSTM(16, activation='tanh', input_shape=(1, X_train.shape[1])), # 16 neurons, standard tanh activation
        Dropout(0.05), # Dropout of 5% neurons during training to prevent overfitting and prevent model from learning too much from training dataset
        Dense(8, activation='relu'), # Hidden layer with 8 neurons and relu activation function
        Dense(1) # Output layer with 1 neuron
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse') # model tries to minimise the mse, i.e, square of the difference b/w actual and predicted values

    # Early stopping to prevent overfittting with patience level of 20, i.e. if the loss doesnt improve in 20 epochs the training stops to save time.
    early_stop = EarlyStopping(
        monitor='loss', patience=20, restore_best_weights=True
    )

    # Fit the model training data
    # No validation set because the dataset is less.
    model.fit(X_train_3D, y_train_scaled,
              epochs=200, # 200 epochs
              batch_size=2, # The model updates its weights after looking at every 2 rows of data.
              verbose=0,
              callbacks=[early_stop])

    # Predict on train data
    train_pred_scaled = model.predict(X_train_3D)
    train_pred_real = np.exp(scaler_y.inverse_transform(train_pred_scaled).flatten())  # Transform prediction results back to the original scale
    y_train_real = np.exp(y_train.values) # Transform actual results back to the original scale
    train_years = pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values # Extract years for training set

    # Predict on test data
    test_pred_scaled = model.predict(X_test_3D)
    test_pred_real = np.exp(scaler_y.inverse_transform(test_pred_scaled).flatten()) # Transform prediction results back to the original scale
    y_test_real = np.exp(y_test.values) # Transform actual results back to the original scale
    test_years = post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values # Extract years for test set

    # We pull the years directly from the dataframes used to split your data
    train_years = pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values
    test_years = post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values

    # Return Train DataFrame
    df_train_lstm = pd.DataFrame({
        'Country': country_name,
        'Year': train_years,
        'Actual_BEV_%': y_train_real,
        'LSTM_Pred': train_pred_real,
        'Type': 'Train'
    })

    # Return Test DataFrame
    df_test_lstm =  pd.DataFrame({
        'Country': country_name,
        'Year': post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_test_real,
        'LSTM_Pred': test_pred_real
    })

    return pd.concat([df_train_lstm, df_test_lstm], ignore_index=True)

**2. ARIMAX Model**

In [16]:
def run_arimax_model(X_train, y_train, X_test, y_test, country_name):

    # Auto-select AR and MA order
    # where P "Autoregressive": "How many past years should we look at to predict the next year"
    # where q "Moving Average": "How many past errors/trends should the model consider"
    try:
        order_res = arma_order_select_ic(y_train, ic='aic', max_ar=2, max_ma=2)

        # aic: Akaike Information Criterion: A statistical estimator of prediction error between p and q.
        # It tries different combinations of p and q, and picks the combination with the lowest aic.
        # Lower the aic better the prediction. It a point where the model is neither overfitting noe underfitting
        p, q = order_res.aic_min_order
    except Exception:
        p, q = 1, 0  # fallback meaning that if the "try" block fails it will initialise p=1 and q=0

    # Train the model on the X-ogenous pca features
    # 1=d(Integrated) component, which subtracts the previous year from the current year to make the data stable before analyzing it.
    model = ARIMA(y_train, exog=X_train, order=(p, 1, q)).fit()

    # Predict on test data
    train_pred_log = model.predict(start=0, end=len(y_train)-1, exog=X_train)
    train_p_real = np.exp(train_pred_log) # Transform prediction results back to the original scale
    y_train_real = np.exp(y_train) # Transform prediction results back to the original scale
    train_years = pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values # Extract training years

    # Predict on test data
    test_pred_log = model.forecast(steps=len(y_test), exog=X_test)
    test_p_real = np.exp(test_pred_log) # Transform prediction results back to the original scale
    y_test_real = np.exp(y_test) # Transform prediction results back to the original scale
    test_years = post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values # Extract testing years

    # Return Train DataFrame
    df_train_arimax = pd.DataFrame({
        'Country': country_name,
        'Year': train_years,
        'Actual_BEV_%': y_train_real.values,
        'ARIMAX_Pred': train_p_real.values,
        'Type': 'Train'
    })

    # Return Test DataFrame
    df_test_arimax = pd.DataFrame({
        'Country': country_name,
        'Year': post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_test_real.values,
        'ARIMAX_Pred': test_p_real.values
    })

    return pd.concat([df_train_arimax, df_test_arimax], ignore_index=True)

In [17]:
# Combining leaders and followers datasets
all_meta_input = []

for country in (leaders_list + followers_list):

    # Data for the specific country to train and test datframe
    # X_tr = train using features from leaders + followers, y_tr = target of leaders + followers
    # X_te = test on using features from leaders + followers, y_te = target of leaders + followers
    X_tr = X_train_pca[pre_covid_fixed['Country'] == country].reset_index(drop=True)
    y_tr = y_train[pre_covid_fixed['Country'] == country].reset_index(drop=True)
    X_te = X_test_pca[post_covid_fixed['Country'] == country].reset_index(drop=True)
    y_te = y_test[post_covid_fixed['Country'] == country].reset_index(drop=True)

    # Run both models
    df_arimax = run_arimax_model(X_tr, y_tr, X_te, y_te, country)
    df_lstm = run_lstm_model(X_tr, y_tr, X_te, y_te, country)

    # Merge and store the prediction results for as per "Country" and "Year"
    merged = pd.merge(df_arimax, df_lstm[['Country', 'Year', 'LSTM_Pred']], on=['Country', 'Year'])
    all_meta_input.append(merged)

# Prepare the input data for Meta Model
meta_dataset = pd.concat(all_meta_input, ignore_index=True)

# Label all the rows with "Year" = 2020-2024 as 'Test', else 'Train'
meta_dataset['Type'] = np.where(meta_dataset['Year'] >= 2020, 'Test', 'Train')

# Add labels to the countries if it belongs to the Leaders or Followers group
meta_dataset['Cluster'] = np.where(meta_dataset['Country'].isin(leaders_list), 'Leaders', 'Followers')

# Sort in alphabetical order and year-wise
cols = ['Country', 'Year', 'Actual_BEV_%', 'Type', 'ARIMAX_Pred', 'LSTM_Pred', 'Cluster']
meta_dataset = meta_dataset[cols].sort_values(['Country', 'Year']).reset_index(drop=True)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: Conver

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: Conver

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 397ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: Conver

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: Conver

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: Conver

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step


In [18]:
meta_dataset.head()

,Country,Year,Actual_BEV_%,Type,ARIMAX_Pred,LSTM_Pred,Cluster
0,Austria,2011,1.18,Train,1.763358,1.187818,Leaders
1,Austria,2012,1.13,Train,1.308595,1.173777,Leaders
2,Austria,2013,1.21,Train,0.902587,1.365151,Leaders
3,Austria,2014,1.44,Train,1.580887,1.401796,Leaders
4,Austria,2015,1.55,Train,1.508027,1.944269,Leaders


**3. META-MODEL (ElasticNet)**

In [19]:
from sklearn.linear_model import ElasticNetCV

final_results_list = []

for cluster in meta_dataset['Cluster'].unique():
    cluster_df = meta_dataset[meta_dataset['Cluster'] == cluster].copy()
    train = cluster_df[cluster_df['Type'] == 'Train']
    if train.empty: continue

    # Initialise ElasticNet
    model_enet = ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
        alphas=np.logspace(-3, 3, 20),
        cv=3,
        positive=True,
        max_iter=50000
    )

    # Fit on ARIMAX and LSTM predictions
    model_enet.fit(train[['ARIMAX_Pred', 'LSTM_Pred']], train['Actual_BEV_%'])

    # Predict for each country in the cluster
    for country in cluster_df['Country'].unique():
        country_df = cluster_df[cluster_df['Country'] == country].copy()
        test = country_df[country_df['Type'] == 'Test']
        if test.empty: continue

        pred = model_enet.predict(test[['ARIMAX_Pred', 'LSTM_Pred']])[0]

        res = test.copy()
        res['Stacked_Pred'] = np.clip(pred, 0, 100)
        res['Error_Abs'] = abs(res['Actual_BEV_%'] - res['Stacked_Pred'])
        final_results_list.append(res)

# Combine results into one DataFrame
meta_results = pd.concat(final_results_list, ignore_index=True)


In [20]:
all_leaders = []
all_followers = []

for year in sorted(meta_results['Year'].unique()):
    year_data = meta_results[meta_results['Year'] == year]

    # Leaders
    leaders_data = year_data[year_data['Cluster'] == 'Leaders'] \
        .sort_values(['Country', 'Year'], ascending=[True, True])

    # Followers
    followers_data = year_data[year_data['Cluster'] == 'Followers'] \
        .sort_values(['Country', 'Year'], ascending=[True, True])

    # Store
    all_leaders.append(leaders_data)
    all_followers.append(followers_data)

# Combine all years and sort by Country then Year
leaders_final = pd.concat(all_leaders).sort_values(['Country', 'Year'], ascending=[True, True])
followers_final = pd.concat(all_followers).sort_values(['Country', 'Year'], ascending=[True, True])

In [21]:
# Save Excel files
leaders_final.to_excel("Trial1_Proposed (MM-ElasticNet)_Leaders_Results.xlsx", index=False)
followers_final.to_excel("Trial1_Proposed (MM-ElasticNet)_Followers_Results.xlsx", index=False)